# 89. US-Net and the method on the ViT, label on every width

Two branches, one per card, in a single session of roughly three to five
hours. Nothing in this notebook needs editing: run it as it is.

VA and VB again with one change to the base: every student also takes CE on the hard label at weight 1.0, as Scala does. VA (35.62) sat far under HydraViT† (49.41) on the same slicing; local 8-epoch runs traced it to students trained on KD alone, and the label on every width lifted width 1.0 from 8.3 to 17.9 at epoch 8. One branch per card, seed 1995. This notebook clones
the branch `vit`, not `nhan`. Every row of the ViT table runs on the same
ViT and recipe as notebook 84 (VA, VB); only the method differs.

| | |
|---|---|
| VG | VA (US-Net†) with the label on every width |
| VH | VB (the method) with the label on every width |

**How to read it.** Two rules, set before the run, on the k/8 grid mean. VG near VD (Scala†, 55.23) and VE (HydraViT†, 49.41) says the base is fixed; VG still near 35 says it is not, and VH then reads nothing. VH above VG says the ViT is worth extending; otherwise it stays future work. The vs A and vs K columns of the last table
compare with the ResNet-50 runs BX and DD and mean nothing here: ignore
them.

## Timing

As VA and VB: about 4 to 5 hours for VG and 6 to 7 for VH on a T4. One session.

## `vg_a_label_vit`

VA (US-Net on the ViT, US-Net-dagger) with the label on every width: each student takes CE on the hard label at weight 1.0 beside its inplace distillation, as Scala does. Identical to VA otherwise. Why: VA read 35.62 on the k/8 grid, 38.65 at width 1.0 against HydraViT-dagger's 53.76 on the same slicing. Local 8-epoch runs traced it to students trained on KD alone: under mixup and label smoothing the widest stays near uniform for many epochs, and the KD-only students drag the shared prefix. At epoch 8, width 1.0: VA 8.3 restricted to the grid (so not the partial heads), 17.9 with the label on every width; width 0.25 went from 5.8 to 15.7. Read against VD (Scala-dagger, 55.23) and VE (HydraViT-dagger, 49.41): the base is fixed only if VG lands near them. VH is read against VG.

## `vh_k_heads4_label_vit`

VB (SlimOT on the ViT, as DO) with the label on every width, the same change as VG makes to VA: each student takes CE on the hard label at weight 1.0. Identical to VB otherwise. VB collapsed (15.50 on the k/8 grid), but it ran on the same broken base as VA, so it has to be retested on the fixed one. Locally VB with the label rose at both ends from epoch 9 to 10 (width 1.0 13.6 to 16.8, width 0.25 8.4 to 11.0), where VB fell at the narrow end from epoch 6. Pilot rule, set before the run: VH above VG on the k/8 grid mean. If it holds, the ViT is worth extending; if not, the ViT stays future work.


## Before you start

* Accelerator **GPU T4 x2**, Internet **On**
* Add the CIFAR-100 dataset as an input
* **Save Version -> Save & Run All (Commit)**, not the interactive run.
  An interactive session ends when the browser closes.

The checks at the top run on the CPU and stop the session in about two
minutes if anything is wrong, before a card is touched. Note that the
branch suite mimics the training loop rather than running it, so for
these branches the smoke step below is what actually exercises the new
code. Do not skip it.

If the session times out partway, carry it forward rather than starting
over. Every epoch writes a checkpoint, so at most one is lost, and the
learning-rate schedule is restored with it - resuming does not restart
the cosine.

1. Save Version on the timed-out session, so its output persists.
2. Make a fresh copy of this notebook.
3. Add data -> Notebook Output -> pick that session.
4. Set `RESUME_FROM` to the directory that *contains* `cifar100_<branch>`,
   then Save & Run All.

That directory is in one of two places depending on how the session
ended, because the copy at the bottom of this notebook only runs if the
session got that far:

    finished the last cell   /kaggle/input/<slug>/logs
    killed mid-training      /kaggle/input/<slug>/new-pruning/logs

If in doubt, run `!find /kaggle/input -name latest_checkpoint.pt` in a
scratch cell and take the parent of the parent. A correct path prints
`restored cifar100_<branch>` and then `Loaded checkpoint ... at epoch N`;
if you see `starting from scratch` instead, the path is wrong and the run
will silently begin again at epoch 1.

## When it finishes

Send back the final table. Pasting the output of the last cell is enough.

In [ ]:
# Fixed for this notebook. Notebook 89 of 89.
BRANCHES = ['vg_a_label_vit', 'vh_k_heads4_label_vit']

SMOKE_FIRST = True

REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'vit'

CIFAR_DIR = ('/kaggle/input/datasets/nlnk1607/cifar100/cifar-100-python')

# To carry a timed-out session forward, attach its output and name the
# logs directory. Leave empty to start fresh.
RESUME_FROM = ''

In [ ]:
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time

import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
for i in range(n_gpu):
    print('  {}: {}'.format(i, torch.cuda.get_device_properties(i).name))

WORK = '/kaggle/working'
CODE = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(CODE):
    subprocess.run(
        ['git', 'clone', '-b', REPO_BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)
print('code at', subprocess.run(
    ['git', 'log', '-1', '--format=%h %s'], capture_output=True,
    text=True).stdout.strip())

available = sorted(
    name[len('cifar100_'):-len('.yml')]
    for name in os.listdir('apps')
    if name.startswith('cifar100_') and name.endswith('.yml'))
print('\nbranches in apps/:')
for name in available:
    print('   ', name)

missing = [b for b in BRANCHES if b not in available]
if missing:
    raise SystemExit('no config for {}'.format(missing))
if n_gpu == 0:
    raise SystemExit('No GPU in this session. Set Accelerator to '
                     'GPU T4 x2 in the notebook settings and run again.')
if n_gpu < len(BRANCHES):
    print('\n{} branches, {} gpu(s): they will run in sequence.'.format(
        len(BRANCHES), n_gpu))

In [ ]:
# What the chosen branches actually differ in, read off the configs
# rather than from the table above, which can drift.
AXES = ('kd_loss', 'cost_source', 'feature_kd', 'feature_align',
        'feature_layers', 'feature_weight', 'tier_weights',
        'horizontal_kd', 'horizontal_where', 'horizontal_loss',
        'horizontal_weight', 'weight_schedule')

settings = {}
for branch in BRANCHES:
    found = {}
    with open('apps/cifar100_{}.yml'.format(branch)) as handle:
        for line in handle:
            key = line.split(':')[0].strip()
            if key in AXES:
                found[key] = line.split(':', 1)[1].strip()
    settings[branch] = found

print('{:20}'.format('') + ''.join(
    '{:>26}'.format(b) for b in BRANCHES))
for axis in AXES:
    values = [settings[b].get(axis, '-') for b in BRANCHES]
    if all(v == '-' for v in values):
        continue
    print('{:20}'.format(axis) + ''.join(
        '{:>26}'.format(v) for v in values))

In [ ]:
TARGET = 'data/cifar-100-python'
if not os.path.isdir(TARGET):
    source = CIFAR_DIR if os.path.isdir(CIFAR_DIR) else None
    if source is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'cifar-100-python' in dirs:
                source = os.path.join(root, 'cifar-100-python')
                break
    os.makedirs('data', exist_ok=True)
    if source:
        os.symlink(source, TARGET)
        print('linked', source)
    else:
        from torchvision import datasets
        datasets.CIFAR100(root='data', train=True, download=True)
        datasets.CIFAR100(root='data', train=False, download=True)
print(sorted(os.listdir(TARGET)))

## Checks, before a card is touched

Seconds on the CPU. Between them these suites have caught a Sinkhorn solved too loosely to have a correct gradient, an alpha-divergence that destroyed the weights in three steps, a calibration that reset the batch norm statistics and never refilled them, and a feature cost normalized so that its own gradient vanished. Every one of those was silent.

The last suite builds each branch in this notebook and runs two training steps of the real loop, profiling included. Four feature branches once reached Kaggle, passed every loss check, and died in the profiler on the first forward, because nothing local had ever called it.

In [ ]:
# Driven by BRANCHES, not by a list written beside it. The branch suite
# used to name its two branches literally, which quietly checked the wrong
# pair for anyone who changed BRANCHES and nothing else.
suites = ['tests/test_loss_ops.py', 'tests/test_bn_calibration.py',
          'tests/test_kd_variants.py',
          'tests/test_channel_reorder.py',
          'tests/test_published_methods.py',
          'tests/test_freeze.py',
          'tests/test_head_groups.py',
          'tests/test_affine_knots.py',
          'tests/test_no_kd.py',
          'tests/test_pre_relu.py',
          'tests/test_dynas.py',
          'tests/test_feature_start.py',
          'tests/test_amp.py',
          'tests/test_wkd.py',
          'tests/test_stratified.py',
          'tests/test_lora.py',
          'tests/test_sam.py']
for suite in suites:
    done = subprocess.run([sys.executable, suite], capture_output=True,
                          text=True)
    print('{:34} {}'.format(
        suite, (done.stdout.strip().splitlines() or ['no output'])[-1]))
    if done.returncode != 0:
        print(done.stdout[-3000:], done.stderr[-2000:])
        raise SystemExit('{} failed'.format(suite))

done = subprocess.run(
    [sys.executable, 'tests/test_all_branches.py'] + BRANCHES,
    capture_output=True, text=True)
print()
print(done.stdout.strip()[-2000:])
if done.returncode != 0:
    print(done.stderr[-2000:])
    raise SystemExit('a branch in {} does not build'.format(BRANCHES))

In [ ]:
if not RESUME_FROM:
    found = set()
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        if ('latest_checkpoint.pt' in files and
                os.path.basename(root)[len('cifar100_'):] in BRANCHES):
            found.add(os.path.dirname(root))
    if len(found) > 1:
        raise SystemExit('checkpoints in more than one input, set '
                         'RESUME_FROM to one of: {}'.format(sorted(found)))
    if found:
        RESUME_FROM = found.pop()
        print('RESUME_FROM was empty; found checkpoints in', RESUME_FROM)
if RESUME_FROM:
    os.makedirs('logs', exist_ok=True)
    for name in os.listdir(RESUME_FROM):
        src = os.path.join(RESUME_FROM, name)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join('logs', name),
                            dirs_exist_ok=True)
            print('restored', name)
else:
    print('starting from scratch')
for branch in BRANCHES:
    ckpt = os.path.join('logs', 'cifar100_' + branch,
                        'latest_checkpoint.pt')
    print(branch, 'resumes from its checkpoint' if os.path.exists(ckpt)
          else 'starts at epoch 1')

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs, quiet=True):
    """one job per card, output interleaved and tagged"""
    lines = queue.Queue()
    procs = {}
    failing = set()

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for index, (label, config) in enumerate(jobs):
        env = dict(os.environ)
        env['CUDA_VISIBLE_DEVICES'] = str(index % max(n_gpu, 1))
        proc = subprocess.Popen(
            [sys.executable, '-u', 'train.py', 'app:' + config],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=env)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] started on gpu {} with {}'.format(
            label, env['CUDA_VISIBLE_DEVICES'], config), flush=True)

    started = time.time()
    remaining = len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[float(width)] = (
                float(loss), float(top1))
        # Once a branch starts printing a traceback, stop filtering
        # it. The body is indented, so the rule below would keep the
        # exception and throw away where it came from.
        if 'Traceback (most recent call last)' in line:
            failing.add(label)
        if quiet and label not in failing and (
                line.startswith(('  ', ')', 'Model(', 'Total', 'Item'))
                or not line.strip()):
            continue
        print('[{}] {}'.format(label, line), flush=True)

    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
if SMOKE_FIRST:
    codes = run_pinned(
        [(b, 'apps/smoke_{}.yml'.format(b)) for b in BRANCHES])
    failed = [b for b, code in codes.items() if code != 0]
    if failed:
        raise SystemExit('smoke failed for {}'.format(failed))
    for b in BRANCHES:
        shutil.rmtree('logs/smoke_{}'.format(b), ignore_errors=True)
    results.clear()
    print('\nsmoke ok')

In [ ]:
codes = run_pinned(
    [(b, 'apps/cifar100_{}.yml'.format(b)) for b in BRANCHES])
print(codes)

## Results, against A and K

A is the number to beat, not C or D. Improving on an ablation of your own
method is not improving on the paper.

A here is BX, A on ResNet-50 at seed 1995 (mean 76.86). The same config
run twice at seed 2026 gave 76.46 and 76.07, so a gap under about 0.4
is not yet a result.

K here is DD, K as kept (feature transport from epoch 6) at seed 1995
(mean 77.50). Every branch in this notebook is a change to K, so "vs K"
is the column that says whether the change helps.

In [ ]:
# BX: A on ResNet-50, seed 1995, for reference. Not the ResNet-18 A,
# which every ResNet-50 run beats by two points without trying.
A_KL = {0.25: 75.26, 0.30: 75.35, 0.35: 75.54, 0.40: 76.01, 0.45: 76.30,
        0.50: 76.77, 0.55: 76.63, 0.60: 77.03, 0.65: 77.13, 0.70: 77.20,
        0.75: 77.39, 0.80: 77.56, 0.85: 77.60, 0.90: 77.99, 0.95: 78.01,
        1.00: 78.04}

# DD: K on ResNet-50 as kept (feature transport from epoch 6), seed 1995
K_REF = {0.25: 75.63, 0.30: 76.29, 0.35: 76.54, 0.40: 76.97, 0.45: 77.67,
         0.50: 77.43, 0.55: 77.58, 0.60: 77.62, 0.65: 77.83, 0.70: 78.02,
         0.75: 78.00, 0.80: 78.01, 0.85: 78.09, 0.90: 78.23, 0.95: 78.08,
         1.00: 77.95}

widths = sorted({w for table in results.values() for w in table})
header = '{:>7}{:>8}{:>8}'.format('width', 'A', 'K')
for branch in BRANCHES:
    header += '{:>11}{:>7}{:>7}'.format(branch[:10], 'vs A', 'vs K')
print(header)

for width in widths:
    row = '{:>7.2f}{:>8.2f}{:>8.2f}'.format(
        width, A_KL.get(width, float('nan')), K_REF.get(width, float('nan')))
    for branch in BRANCHES:
        entry = results.get(branch, {}).get(width)
        if entry is None:
            row += '{:>11}{:>7}{:>7}'.format('-', '-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>11.2f}{:>+7.2f}{:>+7.2f}'.format(
            accuracy, accuracy - A_KL.get(width, accuracy),
            accuracy - K_REF.get(width, accuracy))
    print(row)

print()
reference = sum(A_KL.values()) / len(A_KL)
k_reference = sum(K_REF.values()) / len(K_REF)
print('{:22} mean {:.2f}   worst {:.2f}'.format(
    'A (BX)', reference, min(A_KL.values())))
print('{:22} mean {:.2f}   worst {:.2f}'.format(
    'K (DD)', k_reference, min(K_REF.values())))
for branch in BRANCHES:
    table = results.get(branch, {})
    if not table:
        continue
    accuracies = [100.0 * (1.0 - v[1]) for v in table.values()]
    mean = sum(accuracies) / len(accuracies)
    above_k = sum(100.0 * (1.0 - v[1]) > K_REF.get(round(w, 2), 1e9)
                  for w, v in table.items())
    print('{:22} mean {:.2f}   worst {:.2f}   vs A {:+.2f}   vs K {:+.2f}'
          '   above K at {}/{} widths'.format(
              branch, mean, min(accuracies), mean - reference,
              mean - k_reference, above_k, len(table)))

out = os.path.join(WORK, 'logs')
for branch in BRANCHES:
    log_dir = 'logs/cifar100_{}'.format(branch)
    if os.path.isdir(log_dir):
        shutil.copytree(log_dir, os.path.join(out, 'cifar100_' + branch),
                        dirs_exist_ok=True)
print('\ncheckpoints copied to', out)